In [9]:
!pip install plotly scipy scikit-learn statsmodels -q

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, f_oneway
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from google.colab import files

# Configure Plotly to render seamlessly within Google Colab environment
import plotly.io as pio
pio.renderers.default = "colab"

class PlottingMethods:
    """
    Utility class designed for generating granular graphical components
    and returning them as raw HTML strings for embedding capabilities.
    """

    def bar_chart(self, df, column):
        """Generates an advanced bar chart displaying absolute frequencies and relative percentages."""
        value_counts = df[column].value_counts().reset_index()
        value_counts.columns = ["Category", "Count"]
        total_records = value_counts["Count"].sum()
        value_counts['Percentage'] = ((value_counts['Count'] / total_records) * 100).round(2)

        labels = value_counts.apply(lambda row: f"{row['Count']} ({row['Percentage']}%)", axis=1)
        fig = px.bar(
            value_counts, x="Category", y="Count",
            text=labels,
            title=f"Distribution Analysis - {column}"
        )
        return fig.to_html()

    def pie_chart(self, df, column):
        """Generates an interactive Pie Chart to visualize categorical composition."""
        freq_data = df[column].value_counts().reset_index()
        freq_data.columns = ["Category", "Count"]
        fig = px.pie(freq_data, names="Category", values="Count", title=f"Composition Matrix - {column}")
        return fig.to_html()

    def histogram(self, df, column):
        """Generates a standard continuous Frequency Histogram."""
        fig = px.histogram(df, x=column, title=f"Continuous Frequency Histogram - {column}")
        return fig.to_html()


class DataInspector:
    """
    Main pipeline module engineered to ingest, profile, clean,
    preprocess, and evaluate statistical correlations across multi-type features.
    """

    def __init__(self):
        self.data_frame = None
        self.chart_engine = PlottingMethods()

    def upload_data(self):
        """Handles local file ingestion and automatically sanitizes common missing value flags."""
        user_uploads = files.upload()
        if not user_uploads:
            print("Process terminated: No input data stream detected.")
            return None

        target_file = list(user_uploads.keys())[0]
        garbage_markers = ['?', 'n/a', 'NULL', ' ', 'NA', 'NaN']
        self.data_frame = pd.read_csv(target_file, na_values=garbage_markers)

        # Internal logical verification to cast incorrectly parsed numeric objects
        for col_name in self.data_frame.columns:
            numeric_cast = pd.to_numeric(self.data_frame[col_name], errors='coerce')
            if not numeric_cast.isna().all():
                self.data_frame[col_name] = numeric_cast

        print(f"Data ingestion successful. Current structural shape: {self.data_frame.shape[0]} rows x {self.data_frame.shape[1]} columns.")
        return self.data_frame

    def data_summary(self):
        """Profiles the high-level structural components and schema configuration of the dataset."""
        if self.data_frame is None:
            print("Execution halted: Structural matrix is null.")
            return

        print(f"Total Observational Rows: {self.data_frame.shape[0]} | Matrix Attributes: {self.data_frame.shape[1]}")
        display(self.data_frame.head(20))

        num_fields = self.data_frame.select_dtypes(include=np.number).columns.tolist()
        cat_fields = self.data_frame.select_dtypes(exclude=np.number).columns.tolist()

        print(f"\n[Detected Quantitative Attributes]: {num_fields}")
        print(f"[Detected Qualitative Attributes]: {cat_fields}")

    def handle_missing_values(self, strategy="mean", constant=None):
        """Applies advanced imputation techniques to eliminate incomplete vector occurrences."""
        if self.data_frame is None:
            return

        for column in self.data_frame.columns:
            if self.data_frame[column].isnull().sum() == 0:
                continue

            if strategy == "mean" and pd.api.types.is_numeric_dtype(self.data_frame[column]):
                mean_value = self.data_frame[column].mean()
                self.data_frame[column] = self.data_frame[column].fillna(mean_value)
            elif strategy == "median" and pd.api.types.is_numeric_dtype(self.data_frame[column]):
                median_value = self.data_frame[column].median()
                self.data_frame[column] = self.data_frame[column].fillna(median_value)
            elif strategy == "mode":
                fallback_mode = self.data_frame[column].mode()[0]
                self.data_frame[column] = self.data_frame[column].fillna(fallback_mode)
            elif strategy == "constant" and constant is not None:
                self.data_frame[column] = self.data_frame[column].fillna(constant)

        print(f"Imputation matrix execution complete using routine path: '{strategy}'.")

    def remove_duplicates(self):
        """Detects and prunes perfectly redundant observational instances."""
        if self.data_frame is None:
            return
        initial_count = len(self.data_frame)
        self.data_frame.drop_duplicates(inplace=True)
        print(f"Redundancy purging executed. Purged observation rows: {initial_count - len(self.data_frame)}")

    def handle_outliers(self, column, remove=False):
        """Utilizes the Interquartile Range (IQR) standard to isolate numerical variance anomalies."""
        if self.data_frame is None or column not in self.data_frame.columns:
            return

        percentile_25 = self.data_frame[column].quantile(0.25)
        percentile_75 = self.data_frame[column].quantile(0.75)
        iqr_span = percentile_75 - percentile_25

        floor_limit = percentile_25 - (1.5 * iqr_span)
        ceiling_limit = percentile_75 + (1.5 * iqr_span)

        outlier_mask = (self.data_frame[column] < floor_limit) | (self.data_frame[column] > ceiling_limit)
        outlier_records = self.data_frame[outlier_mask]

        print(f"Feature Vector Evaluation '{column}': Isolated {len(outlier_records)} anomalous records outside IQR boundaries.")
        if remove:
            self.data_frame.drop(outlier_records.index, inplace=True)
            print(f"Data state mutation applied: Anomalous index records dropped successfully.")
        return outlier_records

    def delete_rows(self, indices_str):
        """Enables string-parsed positional index drop sequences for selective row filtration."""
        if self.data_frame is None:
            return
        parsed_indices = [int(token.strip()) for token in indices_str.split(",") if token.strip().isdigit()]
        self.data_frame.drop(index=parsed_indices, inplace=True, errors='ignore')
        print(f"Sub-row administrative elimination applied on locations: {parsed_indices}")

    def delete_columns(self, cols_str):
        """Enables string-parsed token extraction for administrative structural column drops."""
        if self.data_frame is None:
            return
        target_columns = [col.strip() for col in cols_str.split(",") if col.strip() in self.data_frame.columns]
        self.data_frame.drop(columns=target_columns, inplace=True)
        print(f"Structural attribute deletion executed on: {target_columns}")

    def extract_normalized_numeric_data(self, columns, method="minmax"):
        """Transforms spatial scale variance across features using MinMax, Standard, or Robust scaling."""
        if self.data_frame is None:
            return None

        scaler_mapping = {
            "minmax": MinMaxScaler(),
            "standard": StandardScaler(),
            "robust": RobustScaler()
        }

        selected_scaler = scaler_mapping.get(method.lower(), MinMaxScaler())
        transformed_matrix = selected_scaler.fit_transform(self.data_frame[columns])

        generated_headers = [f"{orig_col}_{method}" for orig_col in columns]
        return pd.DataFrame(transformed_matrix, columns=generated_headers, index=self.data_frame.index)

    def extract_normalized_categorical_data(self, columns, method="onehot"):
        """Converts structural string objects into algorithmic representations via alternative encodings."""
        if self.data_frame is None:
            return None

        if method.lower() == "onehot":
            return pd.get_dummies(self.data_frame[columns], columns=columns, drop_first=True, dtype=float)

        elif method.lower() == "ordinal":
            transformer = OrdinalEncoder()
            transformed_matrix = transformer.fit_transform(self.data_frame[columns].astype(str))
            headers = [f"{orig_col}_encoded" for orig_col in columns]
            return pd.DataFrame(transformed_matrix, columns=headers, index=self.data_frame.index)

        elif method.lower() == "uniform":
            transformer = OrdinalEncoder()
            transformed_matrix = transformer.fit_transform(self.data_frame[columns].astype(str))

            # Formulate mathematical uniform scale capping between 0 and 1
            for tracking_idx in range(transformed_matrix.shape[1]):
                max_observed = transformed_matrix[:, tracking_idx].max()
                if max_observed > 0:
                    transformed_matrix[:, tracking_idx] /= max_observed

            headers = [f"{orig_col}_uniform" for orig_col in columns]
            return pd.DataFrame(transformed_matrix, columns=headers, index=self.data_frame.index)

    def merge_datasets(self, numeric_df, categorical_df):
        """Concatenates newly extracted engineering structures horizontally with the master operational matrix."""
        if self.data_frame is None:
            return None
        return pd.concat([self.data_frame, numeric_df, categorical_df], axis=1)

    def plot_univariate_subplots(self, column):
        """Constructs a three-panel statistical layout containing box, scatter, and histogram tracking."""
        if self.data_frame is None or column not in self.data_frame.columns:
            return

        panel = make_subplots(
            rows=3, cols=1,
            subplot_titles=(f"Boxplot Quantile Visualization ({column})", f"Sequential Data-Point Index Spread ({column})", f"Frequency Mass Density ({column})")
        )

        panel.add_trace(go.Box(x=self.data_frame[column], name="Box Structure"), row=1, col=1)
        panel.add_trace(go.Scatter(y=self.data_frame[column], mode='markers', name="Observational Point"), row=2, col=1)
        panel.add_trace(go.Histogram(x=self.data_frame[column], name="Distribution Unit"), row=3, col=1)

        panel.update_layout(height=850, title_text=f"Multi-View Univariate Panel Analysis: {column}", showlegend=False)
        panel.show()

    def plot_relationship(self, col1, col2):
        """Automated dynamic visual script that evaluates mathematical structures to render correct cross-charts."""
        if self.data_frame is None:
            return

        is_col1_numeric = pd.api.types.is_numeric_dtype(self.data_frame[col1])
        is_col2_numeric = pd.api.types.is_numeric_dtype(self.data_frame[col2])

        if is_col1_numeric and is_col2_numeric:
            fig = px.scatter(self.data_frame, x=col1, y=col2, trendline="ols", title=f"Bivariate Scatter Plot Analysis: {col1} vs {col2}")
        elif not is_col1_numeric and is_col2_numeric:
            fig = px.box(self.data_frame, x=col1, y=col2, points="all", title=f"Conditional Box Profile: {col1} Segmented By {col2}")
        elif is_col1_numeric and not is_col2_numeric:
            fig = px.box(self.data_frame, x=col2, y=col1, points="all", title=f"Conditional Box Profile: {col2} Segmented By {col1}")
        else:
            fig = px.histogram(self.data_frame, x=col1, color=col2, barmode="group", title=f"Cross-Nominal Group Contingency: {col1} vs {col2}")
        fig.show()

    def plot_all_associations_heatmap(self):
        """Computes mixed mathematical metrics (Pearson's r, Cramér's V, ANOVA Eta) to render a unified correlation grid."""
        if self.data_frame is None:
            return

        feature_list = self.data_frame.columns.tolist()
        dimension_size = len(feature_list)
        association_matrix = np.zeros((dimension_size, dimension_size))

        for r_idx in range(dimension_size):
            for c_idx in range(dimension_size):
                feat_a, feat_b = feature_list[r_idx], feature_list[c_idx]

                if r_idx == c_idx:
                    association_matrix[r_idx, c_idx] = 1.0
                    continue

                is_a_num = pd.api.types.is_numeric_dtype(self.data_frame[feat_a])
                is_b_num = pd.api.types.is_numeric_dtype(self.data_frame[feat_b])

                try:
                    if is_a_num and is_b_num:
                        association_matrix[r_idx, c_idx] = self.data_frame[[feat_a, feat_b]].corr().iloc[0, 1]
                    elif not is_a_num and not is_b_num:
                        cross_tab = pd.crosstab(self.data_frame[feat_a], self.data_frame[feat_b])
                        if cross_tab.size == 0 or cross_tab.sum().sum() == 0:
                            association_matrix[r_idx, c_idx] = 0
                            continue
                        chi2_val = chi2_contingency(cross_tab)[0]
                        sample_n = cross_tab.sum().sum()
                        phi2 = chi2_val / sample_n
                        rows, columns = cross_tab.shape
                        phi2_corrected = max(0, phi2 - ((columns - 1) * (rows - 1)) / (sample_n - 1))
                        rows_corrected = rows - ((rows - 1) ** 2) / (sample_n - 1)
                        cols_corrected = columns - ((columns - 1) ** 2) / (sample_n - 1)
                        denominator = min((cols_corrected - 1), (rows_corrected - 1))
                        association_matrix[r_idx, c_idx] = np.sqrt(phi2_corrected / denominator) if denominator > 0 else 0
                    else:
                        continuous_var = feat_a if is_a_num else feat_b
                        nominal_var = feat_b if is_a_num else feat_a
                        data_buckets = [frame[continuous_var].dropna().values for _, frame in self.data_frame.groupby(nominal_var)]
                        data_buckets = [bucket for bucket in data_buckets if len(bucket) > 0]
                        if len(data_buckets) > 1:
                            variance_f_score = f_oneway(*data_buckets)[0]
                            normalization_factor = variance_f_score + len(self.data_frame) - len(data_buckets)
                            association_matrix[r_idx, c_idx] = np.sqrt(variance_f_score / normalization_factor) if normalization_factor > 0 else 0
                except:
                    association_matrix[r_idx, c_idx] = 0.0

        cleaned_matrix = np.nan_to_num(association_matrix)
        fig = px.imshow(cleaned_matrix, x=feature_list, y=feature_list, color_continuous_scale='RdBu', aspect="auto",
                        title="Unified Association Heatmap Grid (Pearson / Cramer's V / ANOVA Eta)")
        fig.show()

# =====================================================================
# SYSTEM PIPELINE EXECUTION VERIFICATION
# =====================================================================

print("--- Initializing Production Pipeline Evaluation ---")
evaluator = DataInspector()

# Automatically fetch historical public reference data to validate features without prompt delays
source_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
evaluator.data_frame = pd.read_csv(source_url)

print("\n[Execution Step 1]: Executing Descriptive Structural Profile")
evaluator.data_summary()

print("\n[Execution Step 2]: Executing Vector Missing-Value Imputation")
evaluator.handle_missing_values(strategy="median")

print("\n[Execution Step 3]: Executing Observational Redundancy Scan")
evaluator.remove_duplicates()

print("\n[Execution Step 4]: Executing Continuous Variance Boundary Outlier Scan")
evaluator.handle_outliers("Age", remove=True)

print("\n[Execution Step 5]: Executing Quantitative Scaling and Encoding Operations")
scaled_numeric_features = evaluator.extract_normalized_numeric_data(columns=["Age", "Fare"], method="minmax")
encoded_nominal_features = evaluator.extract_normalized_categorical_data(columns=["Sex", "Embarked"], method="onehot")

# Prevent notebook viewport context truncation on wider dimensions
pd.set_option('display.max_columns', None)

master_merged_set = evaluator.merge_datasets(scaled_numeric_features, encoded_nominal_features)
print("Previewing Engineered Structural Feature Space Matrix:")
display(master_merged_set.head(5))

print("\n[Execution Step 6]: Rendering Complete Analytic Graphics")
evaluator.plot_univariate_subplots("Fare")
evaluator.plot_relationship("Sex", "Survived")
evaluator.plot_all_associations_heatmap()

--- Initializing Production Pipeline Evaluation ---

[Execution Step 1]: Executing Descriptive Structural Profile
Total Observational Rows: 891 | Matrix Attributes: 12


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C



[Detected Quantitative Attributes]: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
[Detected Qualitative Attributes]: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

[Execution Step 2]: Executing Vector Missing-Value Imputation
Imputation matrix execution complete using routine path: 'median'.

[Execution Step 3]: Executing Observational Redundancy Scan
Redundancy purging executed. Purged observation rows: 0

[Execution Step 4]: Executing Continuous Variance Boundary Outlier Scan
Feature Vector Evaluation 'Age': Isolated 66 anomalous records outside IQR boundaries.
Data state mutation applied: Anomalous index records dropped successfully.

[Execution Step 5]: Executing Quantitative Scaling and Encoding Operations
Previewing Engineered Structural Feature Space Matrix:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_minmax,Fare_minmax,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0.372549,0.014151,1.0,0.0,1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,0.686275,0.139136,0.0,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0.450980,0.015469,0.0,0.0,1.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,0.627451,0.103644,0.0,0.0,1.0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0.627451,0.015713,1.0,0.0,1.0



[Execution Step 6]: Rendering Complete Analytic Graphics


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:573: SmallSampleWarning:

all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.

/tmp/ipykernel_939/1763567206.py:291: SmallSampleWarning:

One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:573: SmallSampleWarning:

all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.

/tmp/ipykernel_939/1763567206.py:291: SmallSampleWarning:

One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:573: SmallSampleWarning:

all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.

/tmp/ipykernel_939/1763567206